# POS Tagging and Syntactic Parsing Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: most-frequent-tag baseline

The dumbest POS tagger that works. For each word, predict the tag it had most often in training.

In [ ]:
```python

from collections import Counter, defaultdict

def train_mft(train_examples):

    word_tag_counts = defaultdict(Counter)

    all_tags = Counter()

    for tokens, tags in train_examples:

        for token, tag in zip(tokens, tags):

            word_tag_counts[token.lower()][tag] += 1

            all_tags[tag] += 1

    word_best = {w: c.most_common(1)[0][0] for w, c in word_tag_counts.items()}

    default_tag = all_tags.most_common(1)[0][0]

    return word_best, default_tag

def predict_mft(tokens, word_best, default_tag):

    return [word_best.get(t.lower(), default_tag) for t in tokens]

In [ ]:
```

On the Brown corpus, this baseline hits ~85% accuracy. Not good, but the floor below which no serious model should fall.

### Step 2: bigram HMM tagger

Model the joint probability of the sequence:

In [ ]:
```

P(tags, words) = prod P(tag_i | tag_{i-1}) * P(word_i | tag_i)

In [ ]:
```

Two tables: transition probabilities (tag given previous tag), emission probabilities (word given tag). Estimate both from counts with Laplace smoothing. Decode with Viterbi (dynamic programming over the tag lattice).

In [ ]:
```python

import math

def train_hmm(train_examples, alpha=0.01):

    transitions = defaultdict(Counter)

    emissions = defaultdict(Counter)

    tags = set()

    vocab = set()

    for tokens, ts in train_examples:

        prev = "<BOS>"

        for token, tag in zip(tokens, ts):

            transitions[prev][tag] += 1

            emissions[tag][token.lower()] += 1

            tags.add(tag)

            vocab.add(token.lower())

            prev = tag

        transitions[prev]["<EOS>"] += 1

    return transitions, emissions, tags, vocab

def log_prob(table, given, key, smooth_denom, alpha):

    return math.log((table[given].get(key, 0) + alpha) / smooth_denom)

def viterbi(tokens, transitions, emissions, tags, vocab, alpha=0.01):

    tags_list = list(tags)

    n = len(tokens)

    V = [[0.0] * len(tags_list) for _ in range(n)]

    back = [[0] * len(tags_list) for _ in range(n)]

    for j, tag in enumerate(tags_list):

        em_denom = sum(emissions[tag].values()) + alpha * (len(vocab) + 1)

        tr_denom = sum(transitions["<BOS>"].values()) + alpha * (len(tags_list) + 1)

        tr = log_prob(transitions, "<BOS>", tag, tr_denom, alpha)

        em = log_prob(emissions, tag, tokens[0].lower(), em_denom, alpha)

        V[0][j] = tr + em

        back[0][j] = 0

    for i in range(1, n):

        for j, tag in enumerate(tags_list):

            em_denom = sum(emissions[tag].values()) + alpha * (len(vocab) + 1)

            em = log_prob(emissions, tag, tokens[i].lower(), em_denom, alpha)

            best_prev = 0

            best_score = -1e30

            for k, prev_tag in enumerate(tags_list):

                tr_denom = sum(transitions[prev_tag].values()) + alpha * (len(tags_list) + 1)

                tr = log_prob(transitions, prev_tag, tag, tr_denom, alpha)

                score = V[i - 1][k] + tr + em

                if score > best_score:

                    best_score = score

                    best_prev = k

            V[i][j] = best_score

            back[i][j] = best_prev

    last_best = max(range(len(tags_list)), key=lambda j: V[n - 1][j])

    path = [last_best]

    for i in range(n - 1, 0, -1):

        path.append(back[i][path[-1]])

    return [tags_list[j] for j in reversed(path)]

In [ ]:
```

Bigram HMM on Brown hits ~93% accuracy. The jump from 85% to 93% is mostly transition probabilities — the model learns `DET NOUN` is common and `NOUN DET` is rare.

### Step 3: why modern taggers beat this

Transition + emission probabilities are local. They cannot capture that `saw` is a noun in "I bought a saw" but a verb in "I saw the movie." A CRF with arbitrary features (suffix, word shape, word before and after, word itself) hits ~97%. A BiLSTM-CRF or transformer hits ~98%+.

The ceiling on this task is set by annotator disagreement. Human annotators agree about 97% of the time on Penn Treebank. Models past 98% are probably overfitting the test set.

### Step 4: dependency parsing sketch

Full dependency parsing from scratch is out of scope; the canonical textbook treatment is in Jurafsky and Martin. Two classical families to know:

- **Transition-based** parsers (arc-eager, arc-standard) act like a shift-reduce parser: they read tokens, shift them onto a stack, and apply reduce actions that create arcs. Greedy decoding is fast. Classic implementation is MaltParser. Modern neural version: Chen and Manning's transition-based parser.

- **Graph-based** parsers (Eisner's algorithm, Dozat-Manning biaffine) score every possible head-dependent edge and pick the maximum spanning tree. Slower but more accurate.

For most applied work, call spaCy:

In [ ]:
```python

import spacy

nlp = spacy.load("en_core_web_sm")

doc = nlp("The cats were running at 3pm.")

for token in doc:

    print(f"{token.text:10s} tag={token.tag_:5s} pos={token.pos_:6s} dep={token.dep_:10s} head={token.head.text}")

In [ ]:
```

In [ ]:
```

The        tag=DT    pos=DET    dep=det        head=cats

cats       tag=NNS   pos=NOUN   dep=nsubj      head=running

were       tag=VBD   pos=AUX    dep=aux        head=running

running    tag=VBG   pos=VERB   dep=ROOT       head=running

at         tag=IN    pos=ADP    dep=prep       head=running

3pm        tag=NN    pos=NOUN   dep=pobj       head=at

.          tag=.     pos=PUNCT  dep=punct      head=running

In [ ]:
```

Read the `dep` column bottom to top and the sentence's grammatical structure falls out.

## Exercises

In [ ]:
1. **Easy.** Using the most-frequent-tag baseline on a small tagged corpus (e.g., NLTK's Brown subset), measure accuracy on held-out sentences. Verify the ~85% result.
2. **Medium.** Train the bigram HMM above and report per-tag precision/recall. Which tags does the HMM confuse most?
3. **Hard.** Use spaCy's dependency parse to extract subject-verb-object triples from a 1000-sentence sample. Evaluate on 50 manually labeled triples. Document where extraction fails (often passives, coordinations, and elided subjects).